### 🎯 优化问题的本质

训练一个深度学习模型，本质上是一个**优化问题**：寻找一组模型参数 `θ`，使得损失函数 `L(θ)` 的值最小。

你可以把这个过程想象成在一个复杂、高维、非凸的“山谷”（损失曲面）中寻找最低点。优化算法就是你的“寻路策略”，它决定了你如何根据当前位置的地形（梯度）迈出下一步，以最快的速度到达谷底。



#### 随机梯度下降 (SGD)：一切的起点

SGD 是最基础也最重要的优化算法。它的核心思想非常直观：沿着梯度的反方向（即损失下降最快的方向）更新参数。

*   **更新公式**:
    `θ = θ - η * ∇L(θ)`
    *   `θ`: 模型参数
    *   `η`: 学习率，控制步长
    *   `∇L(θ)`: 损失函数对参数的梯度

*   **深刻洞察**:
    *   **优点**: 简单、高效，通过小批量（Mini-batch）数据估算梯度，大大降低了计算成本。
    *   **缺点**:
        1.  **噪声大**: 小批量梯度是真实梯度的有噪估计，导致更新路径震荡。
        2.  **收敛慢**: 在平坦区域或狭长山谷中，容易陷入停滞或来回震荡。

#### 动量法 (Momentum): 引入“惯性”

为了解决SGD的震荡问题，动量法借鉴了物理学的概念。它不只是看当前的梯度，还保留了之前更新方向的“速度”。

*   **核心思想**:
    想象一个球从山坡滚下，它不仅有重力（当前梯度）的驱动，还有之前滚动的惯性。这让它能更快地冲过平坦地带，并抑制在陡峭侧壁间的震荡。

*   **更新公式**:
    1.  `v = β * v + (1 - β) * ∇L(θ)`  (更新速度)
    2.  `θ = θ - η * v`                  (更新参数)
    *   `v`: 动量（或速度）项，是历史梯度的指数加权平均
    *   `β`: 动量系数，通常设为0.9，控制历史信息的权重

*   **深刻洞察**:
    动量法在相关方向上加速，并抑制震荡，使得收敛路径更加平滑、快速。

#### AdaGrad: 自适应的初步尝试

SGD及其变体对所有参数使用同一个学习率 `η`。但不同参数的重要性、更新频率是不同的。**自适应优化器**的核心思想就是**为每个参数独立地、动态地调整学习率**。

AdaGrad 根据参数的历史梯度大小来调整学习率。对于频繁更新的参数，减小其学习率；对于稀疏更新的参数，增大其学习率。

*   **核心思想**:
    累积所有历史梯度的平方和。如果一个参数的梯度一直很大，其累积值就大，学习率就会被除以一个更大的数，从而变小。

*   **深刻洞察**:
    *   **优点**: 非常适合处理稀疏数据。
    *   **致命缺点**: 梯度平方和会不断累积，导致学习率单调递减，最终趋近于零，训练过早停止。

#### RMSprop: 解决学习率消失

RMSprop 针对 AdaGrad 的缺陷进行了改进。它不再累积所有历史梯度，而是只考虑最近一段时间（一个滑动窗口）的梯度平方，通过指数加权平均来实现。

*   **核心思想**:
    使用梯度平方的移动平均来归一化梯度，避免了学习率无限制下降的问题。

#### Adam: 集大成者

Adam (Adaptive Moment Estimation) 是目前最流行的优化器之一，它巧妙地结合了**动量法**和**RMSprop**的思想。

*   **核心思想**:
    1.  **一阶矩 (m)**: 梯度的移动平均，相当于**动量**。
    2.  **二阶矩 (v)**: 梯度平方的移动平均，相当于**RMSprop**的自适应学习率。

*   **更新流程**:
    1.  计算梯度 `g`
    2.  更新一阶矩 `m = β₁ * m + (1 - β₁) * g`
    3.  更新二阶矩 `v = β₂ * v + (1 - β₂) * g²`
    4.  进行偏差校正（因为初始 `m` 和 `v` 为0，会偏向于0）
    5.  更新参数 `θ = θ - η * m̂ / (√v̂ + ε)`

*   **深刻洞察**:
    Adam 兼具动量法的加速能力和自适应学习率的鲁棒性，对超参数（尤其是学习率）不敏感，通常使用默认值 `lr=0.001` 就能取得不错的效果，是绝大多数任务的**首选默认优化器**。


#### AdamW: 解耦权重衰减

AdamW 是 Adam 的一个极其重要的变体，它修复了 Adam 中权重衰减（L2正则化）实现上的一个缺陷。

*   **问题**: 在标准 Adam 中，L2正则化是通过修改损失函数梯度来实现的。但由于 Adam 的自适应学习率机制，这个修改后的梯度会被错误地缩放，导致权重衰减的效果大打折扣。
*   **解决方案**: AdamW 将**权重衰减**从梯度计算中**解耦**出来，直接在参数更新步骤中减去一部分权重。
    *   `θ = θ - η * (自适应梯度部分 + λ * θ)`
    *   这里的 `λ * θ` 就是直接的权重衰减。

*   **深刻洞察**:
    AdamW 实现了真正的、有效的权重衰减，通常能获得比 Adam 更好的泛化性能。在现代深度学习实践中，**AdamW 已经基本取代了 Adam**。

#### Lion: 追求极致效率

Lion (EvoLved Sign Momentum) 是一个更新颖的优化器，它通过简化操作来提升效率。

*   **核心思想**:
    Lion 只使用**动量**，并且用 `sign` 函数来更新参数。
    *   更新流程：`动量 m = β₁ * m + (1 - β₁) * g` -> `参数更新 θ = θ - η * sign(m)`
    *   `sign(m)` 函数只保留动量的方向（正或负），而忽略其大小。

*   **深刻洞察**:
    1.  **内存效率**: Lion 只需要存储动量 `m`，而 Adam 需要存储 `m` 和 `v`，因此 Lion 的内存占用比 Adam 少约一半。
    2.  **计算效率**: `sign` 操作比 Adam 中的除法运算快得多。
    3.  **性能**: 在许多任务上，Lion 能达到与 AdamW 相当甚至更好的性能，尤其适合大规模模型的训练。

---

#### 总结与选择指南

| 优化器 | 核心思想 | 优点 | 缺点 | 适用场景 |
| :--- | :--- | :--- | :--- | :--- |
| **SGD + Momentum** | 基础梯度下降 + 惯性 | 泛化性能好，内存占用低 | 收敛慢，对学习率敏感 | 计算机视觉（如ResNet），有充足时间调参时 |
| **Adam** | 动量 + 自适应学习率 | 收敛快，对超参数不敏感 | 可能泛化稍差，内存占用高 | 通用默认选择，NLP任务等 |
| **AdamW** | Adam + 解耦权重衰减 | 泛化性能优于Adam，效果稳定 | 内存占用高 | **当前绝大多数任务的默认首选** |
| **Lion** | 符号动量 | 内存和计算效率极高 | 相对较新，生态支持在完善中 | 大规模模型训练，追求训练效率 |

**如何选择？**

1.  **默认选择**: 从 **AdamW** 开始。它通常能提供快速收敛和良好的性能，无需大量调参。
2.  **追求极致泛化**: 如果你有充足的计算资源和时间进行超参数搜索，可以尝试 **SGD + Momentum**，它在某些任务（如图像分类）上可能找到泛化性更好的解。
3.  **大规模训练**: 当模型和显存成为瓶颈时，考虑使用 **Lion** 来节省内存和加速训练。
